In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS olist_gold")
spark.sql("USE olist_gold")
spark.sql("SELECT current_database()").show()

In [0]:

from pyspark.sql.window import Window 

from pyspark.sql.functions import *

In [0]:
master_tbl = spark.table('workspace.olist_silver.master_sales_data')

In [0]:
master_tbl.printSchema()

In [0]:
revenue_by_state = master_tbl.withColumn('month_of_purchase', date_format(col('order_purchase_timestamp') \
    , 'yyyy-MM')).groupBy('customer_state', 'month_of_purchase')\
    .agg(round(sum('total_cost'),2).alias('total_cost'),  count('order_id').alias('order_count'), \
        round(avg('avg_review_score'),2).alias('avg_review_score'))\
    .orderBy('customer_state', 'month_of_purchase')

In [0]:
revenue_by_state.writeTo("workspace.olist_gold.revenue_by_state") \
    .using("delta") \
    .createOrReplace()

In [0]:
delivery_perf = master_tbl.withColumn('month_of_purchase', date_format(col('order_purchase_timestamp'), 'yyyy-MM')) \
    .groupBy('customer_state', 'month_of_purchase') \
    .agg(
        count("order_id").alias('Total_order'),
        count(when(col("Late_delivery") == 'Yes', True)).alias('Late_delivery_count'),
        concat(round(
            (count(when(col("Late_delivery") == 'Yes', True)) / count("order_id")) * 100, 
            2),lit('%'))\
        .alias('Late_delivery_percentage'))\
    .orderBy('month_of_purchase', 'customer_state')

display(delivery_perf)

In [0]:
delivery_perf.writeTo("workspace.olist_gold.delivery_perf") \
    .using("delta") \
    .createOrReplace()

In [0]:
seller_perf = master_tbl.groupBy('seller_id','seller_state','seller_city')\
    .agg(round(sum("total_cost"),2).alias('total_rev'),\
        round(avg('avg_review_score'),2).alias('avg_review_score'),\
        concat(round(count(when(col('late_delivery')== 'Yes',True))\
        /count(col('order_id')) * 100,2),lit('%')).alias('late_delivery_percentage'))
seller_perf.display()

In [0]:
seller_perf.writeTo("workspace.olist_gold.seller_performance") \
    .using("delta") \
    .createOrReplace()